# Solid-Liquid Coexistence Melting-Point Workflow

This notebook contains a technical solid-liquid coexistence workflow for naphthalene using the ALCHEMI Toolkit stack. It is not yet the synchronized toolkit counterpart of the Part 1 AdsorbML adsorption tutorial.

Before using the results scientifically, verify the pinned toolkit commit, run the short smoke settings, and then replace the reduced validation durations with production simulation lengths. The companion planning notes in `OLED-melting-point-case-study.md` still need to be converted into academic markdown cells around the code below.

In [ ]:
import torch
import numpy as np
import csv
import os
import logging
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import matplotlib.pyplot as plt
from ase.io import read as ase_read
from ase import Atoms
from ase.visualize.plot import plot_atoms

from nvalchemi.data import AtomicData, Batch
from nvalchemi.data.datapipes import AtomicDataZarrReader, DataLoader, Dataset
from nvalchemi.dynamics import initialize_velocities, ZarrData
from nvalchemi.dynamics.integrators.nvt_langevin import NVTLangevin
from nvalchemi.dynamics.integrators.npt import NPT
from nvalchemi.dynamics.optimizers.fire2 import FIRE2
from nvalchemi.dynamics.base import ConvergenceHook, DynamicsStage
from nvalchemi.dynamics.hooks import (
    LoggingHook,
    SnapshotHook,
    MaxForceClampHook,
    NaNDetectorHook,
)
from nvalchemi.hooks import NeighborListHook, WrapPeriodicHook
from nvalchemi.models.aimnet2 import AIMNet2Wrapper
from nvalchemi.models.ewald import EwaldModelWrapper
from nvalchemi.models.pipeline import PipelineGroup, PipelineModelWrapper
from nvalchemiops.torch.interactions.electrostatics.parameters import estimate_ewald_parameters

logging.basicConfig(level=logging.INFO)

print(f"NVALCHEMI_TOOLKIT_REF: {os.environ.get('NVALCHEMI_TOOLKIT_REF', 'not set')}")
try:
    print(f"nvalchemi-toolkit version: {version('nvalchemi-toolkit')}")
except PackageNotFoundError:
    print("nvalchemi-toolkit package metadata not found")

In [ ]:
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DT = 0.5                          # fs
FRICTION = 0.001                   # fs^-1 (= 1.0 ps^-1)
T_EQUIL = 100.0                    # K
T_MELT = 800.0                     # K
FMAX = 0.01                        # eV/A
THERMOSTAT_TIME = 1000.0           # fs (NHC, weak coupling)
BAROSTAT_TIME = 1000.0             # fs (MTK)
MAX_FORCE_CLAMP = 50.0             # eV/A

P_1ATM = 101325.0 / 1.602176634e11  # eV/A^3
AMU_OVER_A3_TO_G_CM3 = 1.66054      # 1 amu/A^3 in g/cm^3

# Naphthalene: monoclinic P2_1/a (#14), Z=2
# COD 2100603: a=8.035, b=5.890, c=8.565 A, beta=123.6 deg
# 256 molecules = 128 unit cells; (4,8,4) -> ~32x47x34 A
SUPERCELL = (4, 8, 4)
N_MOL = 256
ATOMS_PER_MOL = 18                 # C10H8
TM_EXP = 353.0                     # K

# 1% durations for pipeline validation (scale up once mechanics confirmed)
THERMALIZE_PS = 0.5                # 1000 steps
EQUILIBRATE_PS = 1.0               # 2000 steps
MELT_PS = 0.5                      # 1000 steps
SLC_PS = 2.0                       # 4000 steps per temperature
SNAPSHOT_EVERY = 100               # steps
LOG_EVERY = 100                    # steps

LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)

TEMPS = [250, 300, 350, 400, 450]

In [ ]:
def compute_density(batch):
    vol = torch.linalg.det(batch.cell).abs().item()
    return batch.atomic_masses.sum().item() * AMU_OVER_A3_TO_G_CM3 / vol


def make_safety_hooks(pipe):
    """Defensive MD hooks (order: neighbor -> wrap -> clamp -> NaN)."""
    return [
        NeighborListHook(pipe.model_config.neighbor_config, stage=DynamicsStage.BEFORE_COMPUTE),
        WrapPeriodicHook(stage=DynamicsStage.AFTER_POST_UPDATE),
        MaxForceClampHook(max_force=MAX_FORCE_CLAMP),
        NaNDetectorHook(),
    ]


def batch_to_ase(batch):
    return Atoms(
        numbers=batch.atomic_numbers.cpu().numpy(),
        positions=batch.positions.detach().cpu().numpy(),
        cell=batch.cell.squeeze().detach().cpu().numpy(),
        pbc=True,
    )


def visualize_structure(batch, title="", save_path=None):
    atoms = batch_to_ase(batch)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, rot, label in zip(
        axes, ["0x,0y,0z", "90x,0y,0z", "0x,90y,0z"], ["xy", "xz", "yz"]
    ):
        plot_atoms(atoms, ax=ax, rotation=rot, show_unit_cell=2)
        ax.set_title(f"{label} view")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def load_zarr_trajectory(zarr_path, device="cpu"):
    """Load a Zarr trajectory into a list of Batch objects."""
    reader = AtomicDataZarrReader(str(zarr_path))
    ds = Dataset(reader, device=device, num_workers=1)
    loader = DataLoader(ds, batch_size=1)
    batches = [b for b in loader]
    ds.close()
    return batches


def plot_trajectory_frames(zarr_path, title="", n_frames=4, save_path=None):
    """Load Zarr trajectory and plot evenly-spaced frames."""
    batches = load_zarr_trajectory(zarr_path)
    n_total = len(batches)
    if n_total == 0:
        print("No frames in trajectory")
        return
    indices = np.linspace(0, n_total - 1, min(n_frames, n_total), dtype=int)
    fig, axes = plt.subplots(1, len(indices), figsize=(5 * len(indices), 5))
    if len(indices) == 1:
        axes = [axes]
    for ax, idx in zip(axes, indices):
        atoms = batch_to_ase(batches[idx])
        plot_atoms(atoms, ax=ax, rotation="0x,0y,0z", show_unit_cell=2)
        ax.set_title(f"Frame {idx}/{n_total}")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


# Custom scalars for LoggingHook
def pressure_scalar(ctx):
    stress = ctx.batch.stress  # [B, 3, 3]
    return -stress.diagonal(dim1=-2, dim2=-1).mean(dim=-1)

def volume_scalar(ctx):
    return torch.linalg.det(ctx.batch.cell).abs().view(-1)

def density_scalar(ctx):
    vol = torch.linalg.det(ctx.batch.cell).abs().view(-1)
    mass_per_graph = torch.zeros(ctx.batch.num_graphs, device=vol.device)
    mass_per_graph.scatter_add_(0, ctx.batch.batch_idx, ctx.batch.atomic_masses)
    return mass_per_graph * AMU_OVER_A3_TO_G_CM3 / vol

NVT_SCALARS = {"pressure_eV_A3": pressure_scalar, "density_g_cm3": density_scalar}
NPT_SCALARS = {"pressure_eV_A3": pressure_scalar, "volume_A3": volume_scalar, "density_g_cm3": density_scalar}


def compute_msd(snapshots, cell, n_atoms_total):
    cell_inv = torch.linalg.inv(cell)
    cumulative_disp = torch.zeros(n_atoms_total, 3, device=snapshots[0].device)
    msd_per_frame = []
    for i in range(1, len(snapshots)):
        dr = snapshots[i] - snapshots[i - 1]
        dr_frac = dr @ cell_inv
        dr_frac -= torch.round(dr_frac)
        dr_mic = dr_frac @ cell
        cumulative_disp += dr_mic
        msd_per_frame.append((cumulative_disp ** 2).sum(dim=-1).cpu())
    return torch.stack(msd_per_frame) if msd_per_frame else torch.zeros(1, n_atoms_total)


def compute_rdf(positions, cell, n_bins=200, r_max=10.0):
    n_atoms = positions.shape[0]
    vol = torch.linalg.det(cell).abs().item()
    rho = n_atoms / vol
    dr = r_max / n_bins
    r_centers = torch.linspace(dr / 2, r_max - dr / 2, n_bins, device=positions.device)
    cell_inv = torch.linalg.inv(cell)
    diff = positions.unsqueeze(0) - positions.unsqueeze(1)
    diff_frac = diff @ cell_inv
    diff_frac -= torch.round(diff_frac)
    diff_cart = diff_frac @ cell
    dists = diff_cart.norm(dim=-1)
    mask = ~torch.eye(n_atoms, dtype=torch.bool, device=positions.device)
    dists = dists[mask]
    hist = torch.histc(dists, bins=n_bins, min=0.0, max=r_max)
    shell_vol = 4 * np.pi * r_centers ** 2 * dr
    g_r = hist / (shell_vol * rho * n_atoms)
    return r_centers.cpu().numpy(), g_r.cpu().numpy()


def compute_S0(positions, atoms_per_mol):
    n_mol = positions.shape[0] // atoms_per_mol
    axes = []
    for m in range(n_mol):
        mol_pos = positions[m * atoms_per_mol : (m + 1) * atoms_per_mol]
        centered = mol_pos - mol_pos.mean(dim=0)
        I = torch.eye(3, device=mol_pos.device) * (centered ** 2).sum() - centered.T @ centered
        _, eigvecs = torch.linalg.eigh(I)
        axes.append(eigvecs[:, 0])
    axes = torch.stack(axes)
    Q = torch.zeros(3, 3, device=positions.device)
    for n in axes:
        Q += (3 * torch.outer(n, n) - torch.eye(3, device=positions.device)) / 2
    Q /= n_mol
    return torch.linalg.eigvalsh(Q)[-1].item()


def read_csv_log(path):
    with open(path) as f:
        return list(csv.DictReader(f))

In [ ]:
unit_cell = ase_read("data/naphthalene.cif")
n_atoms = len(unit_cell)
symbols = unit_cell.get_chemical_symbols()
a, b, c = unit_cell.cell.lengths()
alpha, beta, gamma = unit_cell.cell.angles()

print(f"Atoms: {n_atoms}, C={symbols.count('C')}, H={symbols.count('H')}")
print(f"Cell: a={a:.4f}, b={b:.4f}, c={c:.4f} A, beta={beta:.1f} deg")
print(f"Volume: {unit_cell.cell.volume:.2f} A^3, PBC: {unit_cell.pbc.tolist()}")

expected_elements = {"C", "H"}
assert n_atoms in (18, 36), f"Unexpected atom count: {n_atoms}"
assert set(symbols) == expected_elements, f"Unexpected elements: {set(symbols)}"
print(f"CIF verified: Z={'2' if n_atoms == 36 else '1 (asymmetric unit)'}")

In [ ]:
supercell = unit_cell * SUPERCELL
expected_atoms = N_MOL * ATOMS_PER_MOL
assert len(supercell) == expected_atoms, f"Got {len(supercell)}, expected {expected_atoms}"
print(f"Supercell {SUPERCELL}: {len(supercell)} atoms")

n = len(supercell)
data = AtomicData(
    positions=torch.tensor(supercell.get_positions(), dtype=torch.float32, device=DEVICE),
    atomic_numbers=torch.tensor(supercell.get_atomic_numbers(), dtype=torch.long, device=DEVICE),
    forces=torch.zeros(n, 3, device=DEVICE),
    energy=torch.zeros(1, 1, device=DEVICE),
    cell=torch.tensor(supercell.cell.array, dtype=torch.float32, device=DEVICE).unsqueeze(0),
    pbc=torch.tensor([[True, True, True]], device=DEVICE),
)
batch = Batch.from_data_list([data], device=DEVICE)
batch.atomic_masses = torch.tensor(supercell.get_masses(), dtype=torch.float32, device=DEVICE)
batch.velocities = torch.zeros(n, 3, device=DEVICE)
batch["stress"] = torch.zeros(batch.num_graphs, 3, 3, device=DEVICE)

cell_lengths = batch.cell.squeeze().norm(dim=-1)
print(f"Cell lengths: {[f'{l:.2f}' for l in cell_lengths.tolist()]} A")
print(f"Density: {compute_density(batch):.3f} g/cm3")
assert (cell_lengths > 10.0).all(), "Cell too small for AIMNet2 cutoff"

visualize_structure(batch, title=f"Naphthalene {SUPERCELL} supercell ({n} atoms)",
                    save_path=f"{LOG_DIR}/initial_crystal.png")

In [ ]:
aimnet2 = AIMNet2Wrapper.from_checkpoint("aimnet2", device=DEVICE, compile_model=True)
print(f"AIMNet2 loaded on {DEVICE}, cutoff={aimnet2.model_config.neighbor_config.cutoff} A")

params = estimate_ewald_parameters(batch.positions, batch.cell, batch.batch_idx)
ewald_cutoff = params.real_space_cutoff.max().item()
ewald = EwaldModelWrapper(cutoff=ewald_cutoff, accuracy=1e-6, hybrid_forces=False)
print(f"Ewald cutoff: {ewald_cutoff:.2f} A")

pipe = PipelineModelWrapper(
    groups=[PipelineGroup(steps=[aimnet2, ewald], use_autograd=True)]
)
pipe.set_config("active_outputs", {"energy", "forces", "stress", "charges"})
print(f"Pipeline: {sorted(pipe.model_config.active_outputs)}")

In [ ]:
optimizer = FIRE2(
    model=pipe, dt=0.01, n_steps=5000,
    convergence_hook=ConvergenceHook.from_fmax(threshold=FMAX, source_status=0, target_status=1),
)
for hook in pipe.make_neighbor_hooks():
    optimizer.register_hook(hook)
optimizer.register_hook(MaxForceClampHook(max_force=MAX_FORCE_CLAMP))
optimizer.register_hook(NaNDetectorHook())

with LoggingHook(backend="csv", log_path=f"{LOG_DIR}/minimize.csv", frequency=10) as log:
    optimizer.register_hook(log)
    batch = optimizer.run(batch)

fmax_final = batch.forces.norm(dim=-1).max().item()
print(f"FIRE2: {optimizer.step_count} steps, E={batch.energy.item():.2f} eV, fmax={fmax_final:.6f} eV/A")
assert fmax_final < FMAX, f"Did not converge: fmax={fmax_final}"
print(f"Density: {compute_density(batch):.3f} g/cm3")

In [ ]:
batch.velocities = torch.zeros_like(batch.positions)
initialize_velocities(
    batch.velocities, batch.atomic_masses,
    temperature=torch.tensor([T_EQUIL], device=DEVICE),
    batch_idx=batch.batch_idx, random_seed=42,
    remove_com=True, rescale=True,
)

n_steps = int(THERMALIZE_PS * 1000 / DT)
print(f"Thermalization: {THERMALIZE_PS} ps = {n_steps} steps at {T_EQUIL} K")

zarr_path = Path(LOG_DIR) / "thermalize.zarr"
zarr_sink = ZarrData(store=str(zarr_path), capacity=n_steps)
snap_hook = SnapshotHook(sink=zarr_sink, frequency=SNAPSHOT_EVERY)

nvt = NVTLangevin(
    model=pipe, dt=DT, temperature=T_EQUIL, friction=FRICTION,
    n_steps=n_steps,
    hooks=make_safety_hooks(pipe) + [snap_hook],
)
compiled_run = torch.compile(nvt.run)

with LoggingHook(backend="csv", custom_scalars=NVT_SCALARS,
                 log_path=f"{LOG_DIR}/thermalize.csv", frequency=LOG_EVERY) as log:
    nvt.register_hook(log)
    batch = compiled_run(batch)

print(f"Thermalization done ({nvt.step_count} steps), density={compute_density(batch):.3f} g/cm3")
plot_trajectory_frames(zarr_path, title="Thermalization trajectory", save_path=f"{LOG_DIR}/thermalize_traj.png")

In [ ]:
n_steps = int(EQUILIBRATE_PS * 1000 / DT)
print(f"NPT equilibration: {EQUILIBRATE_PS} ps = {n_steps} steps at {T_EQUIL} K, 1 atm")

zarr_path = Path(LOG_DIR) / "equilibrate.zarr"
zarr_sink = ZarrData(store=str(zarr_path), capacity=n_steps)
snap_hook = SnapshotHook(sink=zarr_sink, frequency=SNAPSHOT_EVERY)

npt = NPT(
    model=pipe, dt=DT, temperature=T_EQUIL, pressure=P_1ATM,
    barostat_time=BAROSTAT_TIME, thermostat_time=THERMOSTAT_TIME,
    pressure_coupling="isotropic", n_steps=n_steps,
    hooks=make_safety_hooks(pipe) + [snap_hook],
)
compiled_run = torch.compile(npt.run)

with LoggingHook(backend="csv", custom_scalars=NPT_SCALARS,
                 log_path=f"{LOG_DIR}/equilibrate.csv", frequency=LOG_EVERY) as log:
    npt.register_hook(log)
    batch = compiled_run(batch)

density = compute_density(batch)
print(f"Equilibration done ({npt.step_count} steps), density={density:.3f} g/cm3")
print(f"Cell lengths: {[f'{l:.2f}' for l in batch.cell.squeeze().norm(dim=-1).tolist()]} A")
plot_trajectory_frames(zarr_path, title="NPT equilibration trajectory", save_path=f"{LOG_DIR}/equilibrate_traj.png")

In [ ]:
crystal_batch = batch.clone()

melt_batch = batch.clone()
initialize_velocities(
    melt_batch.velocities, melt_batch.atomic_masses,
    temperature=torch.tensor([T_MELT], device=DEVICE),
    batch_idx=melt_batch.batch_idx, random_seed=123,
    remove_com=True, rescale=True,
)

n_steps = int(MELT_PS * 1000 / DT)
print(f"Melt generation: {MELT_PS} ps = {n_steps} steps at {T_MELT} K")

zarr_path = Path(LOG_DIR) / "melt.zarr"
zarr_sink = ZarrData(store=str(zarr_path), capacity=n_steps)
snap_hook = SnapshotHook(sink=zarr_sink, frequency=SNAPSHOT_EVERY)

nvt_melt = NVTLangevin(
    model=pipe, dt=DT, temperature=T_MELT, friction=FRICTION,
    n_steps=n_steps,
    hooks=make_safety_hooks(pipe) + [snap_hook],
)
compiled_run = torch.compile(nvt_melt.run)

with LoggingHook(backend="csv", custom_scalars=NVT_SCALARS,
                 log_path=f"{LOG_DIR}/melt.csv", frequency=LOG_EVERY) as log:
    nvt_melt.register_hook(log)
    melt_batch = compiled_run(melt_batch)

print(f"Crystal density: {compute_density(crystal_batch):.3f}, Melt density: {compute_density(melt_batch):.3f} g/cm3")
print(f"Density drop: {(1 - compute_density(melt_batch) / compute_density(crystal_batch)) * 100:.1f}%")
plot_trajectory_frames(zarr_path, title="Melt generation trajectory", save_path=f"{LOG_DIR}/melt_traj.png")

In [ ]:
cell = crystal_batch.cell.squeeze()
c_vec = cell[2, :]

crystal_pos = crystal_batch.positions
melt_pos = melt_batch.positions + c_vec

slc_pos = torch.cat([crystal_pos, melt_pos], dim=0)
slc_Z = torch.cat([crystal_batch.atomic_numbers, melt_batch.atomic_numbers], dim=0)
slc_masses = torch.cat([crystal_batch.atomic_masses, melt_batch.atomic_masses], dim=0)
slc_vel = torch.cat([crystal_batch.velocities, melt_batch.velocities], dim=0)

slc_cell = cell.clone()
slc_cell[2, :] *= 2

n_slc = slc_pos.shape[0]
slc_data = AtomicData(
    positions=slc_pos, atomic_numbers=slc_Z,
    forces=torch.zeros(n_slc, 3, device=DEVICE),
    energy=torch.zeros(1, 1, device=DEVICE),
    cell=slc_cell.unsqueeze(0),
    pbc=torch.tensor([[True, True, True]], device=DEVICE),
)
slc_batch = Batch.from_data_list([slc_data], device=DEVICE)
slc_batch.atomic_masses = slc_masses
slc_batch.velocities = slc_vel
slc_batch["stress"] = torch.zeros(slc_batch.num_graphs, 3, 3, device=DEVICE)

print(f"SLC system: {slc_batch.num_nodes} atoms")
print(f"Cell lengths: {[f'{l:.2f}' for l in slc_batch.cell.squeeze().norm(dim=-1).tolist()]} A")

In [ ]:
n_crystal = crystal_batch.num_nodes
assert slc_batch.num_nodes == 2 * n_crystal

slc_cl = slc_batch.cell.squeeze().norm(dim=-1)
cryst_cl = crystal_batch.cell.squeeze().norm(dim=-1)
assert torch.allclose(slc_cl[:2], cryst_cl[:2], atol=0.01), "a,b changed"
assert abs(slc_cl[2].item() / cryst_cl[2].item() - 2.0) < 0.01, "c not doubled"

# Min distance check (sample for speed)
n = slc_batch.num_nodes
idx = torch.randperm(n, device=DEVICE)[:min(2000, n)]
dists = torch.cdist(slc_batch.positions[idx].unsqueeze(0), slc_batch.positions[idx].unsqueeze(0)).squeeze()
dists.fill_diagonal_(float("inf"))
print(f"Min pairwise distance (sampled): {dists.min().item():.2f} A")
assert dists.min().item() > 0.5

visualize_structure(slc_batch, title=f"SLC system ({slc_batch.num_nodes} atoms)",
                    save_path=f"{LOG_DIR}/slc_construction.png")
print("All SLC checks passed")

In [ ]:
results = {}

for T in TEMPS:
    print(f"\n{'='*50} T = {T} K {'='*50}")

    run_batch = slc_batch.clone()
    run_batch.velocities = torch.zeros(run_batch.num_nodes, 3, device=DEVICE)
    initialize_velocities(
        run_batch.velocities, run_batch.atomic_masses,
        temperature=torch.tensor([float(T)], device=DEVICE),
        batch_idx=run_batch.batch_idx, random_seed=int(T),
        remove_com=True, rescale=True,
    )

    n_steps = int(SLC_PS * 1000 / DT)
    log_path = f"{LOG_DIR}/slc_T{T}K.csv"
    zarr_path = Path(LOG_DIR) / f"slc_T{T}K.zarr"
    zarr_sink = ZarrData(store=str(zarr_path), capacity=n_steps)
    snap_hook = SnapshotHook(sink=zarr_sink, frequency=SNAPSHOT_EVERY)

    npt_slc = NPT(
        model=pipe, dt=DT, temperature=float(T), pressure=P_1ATM,
        barostat_time=BAROSTAT_TIME, thermostat_time=THERMOSTAT_TIME,
        pressure_coupling="isotropic", n_steps=n_steps,
        hooks=make_safety_hooks(pipe) + [snap_hook],
    )
    compiled_run = torch.compile(npt_slc.run)

    with LoggingHook(backend="csv", custom_scalars=NPT_SCALARS,
                     log_path=log_path, frequency=LOG_EVERY) as log:
        npt_slc.register_hook(log)
        run_batch = compiled_run(run_batch)

    results[T] = {
        "final_batch": run_batch,
        "zarr_path": zarr_path,
        "log_path": log_path,
        "density": compute_density(run_batch),
    }
    print(f"T={T}K done ({npt_slc.step_count} steps), density={results[T]['density']:.3f} g/cm3")
    plot_trajectory_frames(zarr_path, title=f"SLC T={T}K", save_path=f"{LOG_DIR}/slc_T{T}K_traj.png")

print(f"\nAll {len(TEMPS)} temperature points complete")

In [ ]:
n_half = slc_batch.num_nodes // 2
analysis = {}

for T in TEMPS:
    res = results[T]
    log_data = read_csv_log(res["log_path"])
    steps = [int(float(r["step"])) for r in log_data]
    energies = [float(r["energy"]) for r in log_data]

    # Load trajectory positions from zarr for MSD
    traj_batches = load_zarr_trajectory(res["zarr_path"], device=DEVICE)
    snap_positions = [b.positions for b in traj_batches]
    mid = len(snap_positions) // 2
    if mid > 1:
        cell_for_msd = res["final_batch"].cell.squeeze()
        msd_data = compute_msd(snap_positions[mid:], cell_for_msd, slc_batch.num_nodes)
        msd_crystal = msd_data[:, :n_half].mean(dim=1)
        msd_melt = msd_data[:, n_half:].mean(dim=1)
    else:
        msd_crystal = msd_melt = torch.zeros(1)

    final_pos = res["final_batch"].positions
    S0_crystal = compute_S0(final_pos[:n_half], ATOMS_PER_MOL)
    S0_melt = compute_S0(final_pos[n_half:], ATOMS_PER_MOL)

    analysis[T] = {
        "steps": steps, "energies": energies,
        "density": res["density"],
        "msd_crystal_final": msd_crystal[-1].item() if len(msd_crystal) > 0 else 0,
        "msd_melt_final": msd_melt[-1].item() if len(msd_melt) > 0 else 0,
        "S0_crystal": S0_crystal, "S0_melt": S0_melt,
    }

print(f"{'T (K)':>8} {'Density':>10} {'MSD_cryst':>10} {'MSD_melt':>10} {'S0_cryst':>10} {'S0_melt':>10}")
print("-" * 68)
for T in TEMPS:
    a = analysis[T]
    print(f"{T:8d} {a['density']:10.3f} {a['msd_crystal_final']:10.2f} "
          f"{a['msd_melt_final']:10.2f} {a['S0_crystal']:10.3f} {a['S0_melt']:10.3f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
for T in TEMPS:
    a = analysis[T]
    ax.plot(a["steps"], a["energies"], label=f"{T} K")
ax.set_xlabel("Step"); ax.set_ylabel("Energy (eV)"); ax.set_title("Potential Energy")
ax.legend()

ax = axes[0, 1]
ax.plot(TEMPS, [analysis[T]["density"] for T in TEMPS], "o-", markersize=8)
ax.axvline(TM_EXP, color="gray", ls="--", label=f"Tm_exp = {TM_EXP} K")
ax.set_xlabel("Temperature (K)"); ax.set_ylabel("Density (g/cm3)"); ax.set_title("Final Density")
ax.legend()

ax = axes[1, 0]
ax.plot(TEMPS, [analysis[T]["msd_crystal_final"] for T in TEMPS], "s-", label="Crystal half")
ax.plot(TEMPS, [analysis[T]["msd_melt_final"] for T in TEMPS], "o-", label="Melt half")
ax.axvline(TM_EXP, color="gray", ls="--", label=f"Tm_exp")
ax.set_xlabel("Temperature (K)"); ax.set_ylabel("MSD (A^2)"); ax.set_title("Mean Squared Displacement")
ax.legend()

ax = axes[1, 1]
ax.plot(TEMPS, [analysis[T]["S0_crystal"] for T in TEMPS], "s-", label="Crystal half")
ax.plot(TEMPS, [analysis[T]["S0_melt"] for T in TEMPS], "o-", label="Melt half")
ax.axvline(TM_EXP, color="gray", ls="--", label=f"Tm_exp")
ax.set_xlabel("Temperature (K)"); ax.set_ylabel("S0"); ax.set_title("Rotational Order Parameter")
ax.legend()

fig.suptitle(f"SLC Analysis - Naphthalene (Tm_exp = {TM_EXP} K)", fontsize=14)
plt.tight_layout()
plt.savefig(f"{LOG_DIR}/slc_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# g(r) at lowest and highest T
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, T in zip(axes, [TEMPS[0], TEMPS[-1]]):
    pos = results[T]["final_batch"].positions
    cell_t = results[T]["final_batch"].cell.squeeze()
    r, gr = compute_rdf(pos, cell_t)
    ax.plot(r, gr)
    ax.set_xlabel("r (A)"); ax.set_ylabel("g(r)"); ax.set_title(f"RDF at T = {T} K")
    ax.set_xlim(0, 10)
plt.tight_layout()
plt.savefig(f"{LOG_DIR}/slc_rdf.png", dpi=150, bbox_inches="tight")
plt.show()